In [2]:
import os
from pathlib import Path

import gspread
import pandas as pd
from google.oauth2.service_account import Credentials
from merge_tables.db.connection import connect_to_postgres_via_duckdb

duck = connect_to_postgres_via_duckdb()

# From URL: https://docs.google.com/spreadsheets/d/<SPREADSHEET_ID>/edit
SPREADSHEET_ID = "1SBXeTGYQrmQXCCDQz21XGJ9iKE0HM-VZP8r69P5TOZ0"
SHEET_NAME = "1.Clientlist"

# credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
credentials_path = "./config/gsheet-creds.json"
# Or set explicitly: credentials_path = Path.home() / "secrets" / "service-account.json"
if not credentials_path:
    raise FileNotFoundError(
        "Set GOOGLE_APPLICATION_CREDENTIALS to your service account JSON path"
    )
credentials_path = Path(credentials_path).expanduser()


# Run the first gspread config cell so `credentials_path`, `SPREADSHEET_ID`, and `SHEET_NAME` exist.


def _sql_literal(s: str) -> str:
    return s.replace("'", "''")


key_path = _sql_literal(str(credentials_path.resolve()))

duck.execute("INSTALL gsheets FROM community;")
duck.execute("LOAD gsheets;")
duck.execute(
    f"""
CREATE OR REPLACE SECRET gsheet_sa (
    TYPE gsheet,
    PROVIDER key_file,
    FILEPATH '{key_path}'
);
"""
)

✓ Successfully connected DuckDB to PostgreSQL database 'medisoft'


In [13]:
UBERREGIONAL_ID = '1UjztKtuYFDzwmjIh6FZk-ihoSSOBrE4f0rehgLk9PeE'
UBERREGIONAL_SHEET = 'basic care - ÜR Kunden'

EM_ID = '1vB84YG3eBVJVAQsv8VNe2K_TTE8bZ1I9gnwidGAvyXE'
EM_SHEET = 'easybill_clients_list'

In [27]:
duck.sql(
    f"""
    select 
        distinct on(uber."easybill ID")
        uber."easybill ID", uber."Kunde", em.medisoft_ids, count(*) over(partition by uber."easybill ID") as c
    from read_gsheet(
        '{UBERREGIONAL_ID}',
        sheet='{UBERREGIONAL_SHEET}',
        all_varchar=true
        ) uber
    left join read_gsheet(
        '{EM_ID}',
        sheet='{EM_SHEET}',
        all_varchar=true
        ) em
            on em.easybill_kundennummer = uber."easybill ID"
    order by 2
    """
).to_csv('uber_kunde.csv')